In [12]:
import numpy as np
import bsf
import esm
import torch

In [30]:
!wget https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
!gunzip Homo_sapiens.GRCh38.pep.all.fa.gz

--2026-07-10 16:13:07--  https://ftp.ensembl.org/pub/release-116/fasta/homo_sapiens/pep/Homo_sapiens.GRCh38.pep.all.fa.gz
Resolving ftp.ensembl.org (ftp.ensembl.org)... 193.62.193.169
Connecting to ftp.ensembl.org (ftp.ensembl.org)|193.62.193.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23319936 (22M) [application/x-gzip]
Saving to: ‘Homo_sapiens.GRCh38.pep.all.fa.gz’

Homo_sapiens.GRCh38 100%[===================>]  22.24M  61.6MB/s    in 0.4s    

2026-07-10 16:13:08 (61.6 MB/s) - ‘Homo_sapiens.GRCh38.pep.all.fa.gz’ saved [23319936/23319936]



In [75]:
def load_fasta_as_dict(fasta_path, max_length=512):
    proteins = {}
    current_header = None
    current_sequence = []

    with open(fasta_path, "r") as fasta_file:
        for line in fasta_file:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                # Save the previous sequence
                if current_header is not None:
                    sequence = "".join(current_sequence)

                    if len(sequence) <= max_length:
                        proteins[current_header] = sequence

                current_header = line[1:]
                current_sequence = []

            else:
                current_sequence.append(line)

        # Save the final sequence
        if current_header is not None:
            sequence = "".join(current_sequence)

            if len(sequence) <= max_length:
                proteins[current_header] = sequence

    return proteins

In [76]:
fasta_path = "../../Homo_sapiens.GRCh38.pep.all.fa"
all_proteins = load_fasta_as_dict(fasta_path)
tenk_proteins = dict(list(all_proteins.items())[:10000])

In [77]:
standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

ten_proteins = {
    protein_id: "".join(
        aa for aa in sequence.upper()
        if aa in standard_amino_acids
    )
    for protein_id, sequence in tenk_proteins.items()
}

In [79]:
# Load ESM-2 model
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
batch_converter = alphabet.get_batch_converter()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

model = model.to(device)

print("model load successfull")



Using: cuda


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 22.06 GiB of which 19.44 MiB is free. Including non-PyTorch memory, this process has 22.03 GiB memory in use. Of the allocated memory 21.59 GiB is allocated by PyTorch, and 176.13 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [74]:
all_data = list(ten_proteins.items())

sequence_representations = []

batch_size = 64
for start in range(0, len(all_data), batch_size):
    print("processing:", start)
    batch_data = all_data[start : start + batch_size]
    batch_labels, batch_strs, batch_tokens = batch_converter(batch_data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)

    batch_tokens = batch_tokens.to(device)


    # Extract per-residue representations (on CPU)
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    token_representations = results["representations"][33]

    # Generate per-sequence representations via averaging
    # NOTE: token 0 is always a beginning-of-sequence token, so the first residue is token 1.

    for i, tokens_len in enumerate(batch_lens):
        sequence_representations.append(token_representations[i, 1 : tokens_len - 1].mean(0))

processing: 0


OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 22.06 GiB of which 19.44 MiB is free. Including non-PyTorch memory, this process has 22.03 GiB memory in use. Of the allocated memory 21.59 GiB is allocated by PyTorch, and 176.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
sequence_representations = torch.stack(sequence_representations)
norm_sequence_representations = sequence_representations - sequence_representations.mean(dim=0)
x = norm_sequence_representations
x = (x / np.sqrt((x**2).sum(1).mean()))*(np.sqrt(x.shape[1]))

In [ ]:
model = bsf.GrassmannianBSF(d=x.shape[1], n_groups=256, group_size=3, l0=8)
bsf.train(model, x, epochs=300, lr=3e-3)